In [ ]:
import cv2
import numpy as np
import os
import glob

def image_to_point_cloud(image_path, output_txt_path, max_points=2000):
    # 1. Wczytanie obrazu w skali szarości
    img = cv2.imread(image_path, cv2.IMREAD_GRAYSCALE)
    
    # 2. Detekcja krawędzi (parametry do dostrojenia empirycznego)
    edges = cv2.Canny(img, threshold1=50, threshold2=150)
    
    # 3. Ekstrakcja współrzędnych pikseli, gdzie krawędź == 255
    y_coords, x_coords = np.where(edges == 255)
    point_cloud = np.column_stack((x_coords, y_coords))
    
    # 4. Downsampling, jeśli punktów jest za dużo dla Ripsera (opcjonalnie)
    if len(point_cloud) > max_points:
        indices = np.random.choice(len(point_cloud), max_points, replace=False)
        point_cloud = point_cloud[indices]
        
    # 5. Zapis do formatu wspieranego przez Ripsera (spacja jako separator)
    np.savetxt(output_txt_path, point_cloud, fmt='%.1f', delimiter=' ')
    return len(point_cloud)

# Przykładowe użycie dla jednego pliku:
img_path = "dataset/train/Normal/sample_01.jpg"
pc_path = "temp_point_cloud.txt"
n_points = image_to_point_cloud(img_path, pc_path)
print(f"Zapisano {n_points} punktów do {pc_path}")

In [ ]:
import subprocess

def run_ripser_cpp(points_file, dim=1, ripser_path="./../../ripser/ripser"):
    log_file = "ripser_output.log"
    stats_file = "ripser_stats.log"
    
    # Budujemy komendę na podstawie Twojego skryptu bashowego
    command = [
        "/usr/bin/time", "-v", 
        ripser_path, 
        "--format", "point-cloud", 
        "--dim", str(dim), 
        points_file
    ]
    
    print("=== STEP 2: Run original Ripser (C++) ===")
    
    # Przekierowanie stdout i stderr
    with open(log_file, "w") as out_f, open(stats_file, "w") as err_f:
        process = subprocess.run(command, stdout=out_f, stderr=err_f, text=True)
        
    if process.returncode == 0:
        print("Ripser zakończył działanie z sukcesem.")
    else:
        print("Błąd wykonania Ripsera.")
        
    return log_file, stats_file

# Wywołanie:
log_file, stats_file = run_ripser_cpp(pc_path)